# PROJETO DE COLETA DE DADOS
## Comparação dos clubes no Brasileirão de 2024 e 2025

**Autor:** Rafael  
**Data da coleta:** 09/11/2026  
**Disciplina:** Introdução à Ciência de Dados

## 1. OBJETIVO
Comparar o desempenho dos clubes no Campeonato Brasileiro de 2024 e 2025. Analisar pontos, gols e aproveitamento, além de identificar quais times melhoraram entre as duas temporadas. Escolhi esse tema porque gosto muito de futebol e achei interessante usar os dados do Brasileirão para comparar o desempenho dos clubes entre duas temporadas.

## 2. FONTES DE DADOS
| Fonte | URL | Método | Dados coletados |
|---|---|---|---|
| ESPN — Brasileirão 2024 | https://www.espn.com.br/futebol/classificacao/_/liga/bra.1/temporada/2024 | Scraping | Classificação e estatísticas dos clubes |
| ESPN — Brasileirão 2025 | https://www.espn.com.br/futebol/classificacao/_/liga/bra.1/temporada/2025 | Scraping | Classificação e estatísticas dos clubes |

## 3. METODOLOGIA
### 3.1 Coleta
Baixei o HTML das duas páginas da ESPN e usei BeautifulSoup para extrair as tabelas. A página separa os nomes dos clubes e as estatísticas em duas tabelas. Combinei as linhas pela posição, conferindo os índices das linhas e a quantidade de registros. Fiz apenas uma consulta por temporada, com intervalo entre as requisições.


In [ ]:
from urllib.request import Request, urlopen
from pathlib import Path
from time import sleep
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

urls = {
    year: f"https://www.espn.com.br/futebol/classificacao/_/liga/bra.1/temporada/{year}"
    for year in [2024, 2025]
}
pages = {}
for year, url in urls.items():
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=60) as response:
        pages[year] = response.read().decode("utf-8")
    sleep(1)
print("Páginas coletadas:", len(pages))

In [ ]:
def extract_table(html, year):
    soup = BeautifulSoup(html, "html.parser")
    if f"{year}" not in soup.title.get_text():
        raise ValueError("A página não corresponde à temporada solicitada.")
    tables = soup.select("table")
    if len(tables) != 2:
        raise ValueError("Estrutura da página mudou: confira as tabelas.")
    teams = tables[0].select("tbody tr")
    statistics = tables[1].select("tbody tr")
    headers = [th.get_text(strip=True) for th in tables[1].select("thead th")]
    if headers != ["J", "V", "E", "D", "GP", "GC", "SG", "PTS"]:
        raise ValueError("As colunas da ESPN mudaram.")
    if len(teams) != 20 or len(statistics) != 20:
        raise ValueError("Quantidade inesperada de clubes.")
    records = []
    for team, stats in zip(teams, statistics):
        assert team.get("data-idx") == stats.get("data-idx")
        name = team.select_one("abbr")
        position = team.select_one(".team-position")
        values = [td.get_text(strip=True) for td in stats.select("td")]
        if len(values) != 8:
            raise ValueError("Linha com estatísticas incompletas.")
        records.append({
            "temporada": year,
            "posicao": position.get_text(strip=True),
            "clube": name["title"],
            **dict(zip(["jogos", "vitorias", "empates", "derrotas",
                        "gols_pro", "gols_contra", "saldo_gols", "pontos"], values))
        })
    return pd.DataFrame(records)

raw_df = pd.concat([extract_table(html, year) for year, html in pages.items()], ignore_index=True)
raw_df.head()

### 3.2 Limpeza
Retirei espaços dos nomes, converti os números extraídos do HTML para inteiros e verifiquei ausências e duplicatas por clube e temporada. Conferi também se vitórias, empates e derrotas somam o total de jogos e se os pontos e o saldo de gols estão corretos.


In [ ]:
df = raw_df.copy()
df["clube"] = df["clube"].str.strip()
for column in df.columns.drop("clube"):
    df[column] = pd.to_numeric(df[column], errors="raise").astype(int)
print("Valores ausentes:", int(df.isna().sum().sum()))
print("Duplicatas por clube e temporada:", int(df.duplicated(["temporada", "clube"]).sum()))
assert not df.isna().any().any()
assert not df.duplicated(["temporada", "clube"]).any()
assert (df["jogos"] == 38).all()
assert (df["vitorias"] + df["empates"] + df["derrotas"] == df["jogos"]).all()
assert (3 * df["vitorias"] + df["empates"] == df["pontos"]).all()
assert (df["gols_pro"] - df["gols_contra"] == df["saldo_gols"]).all()
for year, group in df.groupby("temporada"):
    assert sorted(group["posicao"]) == list(range(1, 21))
    assert group["gols_pro"].sum() == group["gols_contra"].sum()
print("Limpeza concluída")

### 3.3 Processamento
Calculei o aproveitamento como pontos obtidos divididos pelos pontos possíveis, em porcentagem, e a média de gols marcados por jogo. Para comparar a evolução, cruzei os clubes presentes nas duas temporadas pelo nome e calculei a diferença de pontos entre 2024 e 2025.


In [ ]:
df["aproveitamento_pct"] = (df["pontos"] / (df["jogos"] * 3) * 100).round(2)
df["gols_por_jogo"] = (df["gols_pro"] / df["jogos"]).round(2)
comparison = df.pivot(index="clube", columns="temporada", values="pontos").dropna().astype(int)
comparison["diferenca_pontos"] = comparison[2025] - comparison[2024]
comparison = comparison.sort_values("diferenca_pontos", ascending=False)
comparison

## 4. DATASET FINAL
Cada linha representa um clube em uma temporada. O CSV é gerado na pasta em que este notebook for executado.

| Coluna | Tipo | Descrição |
|---|---|---|
| temporada | inteiro | Ano do campeonato |
| posicao | inteiro | Classificação final do clube |
| clube | texto | Nome do clube |
| jogos | inteiro | Jogos disputados |
| vitorias | inteiro | Vitórias |
| empates | inteiro | Empates |
| derrotas | inteiro | Derrotas |
| gols_pro | inteiro | Gols marcados |
| gols_contra | inteiro | Gols sofridos |
| saldo_gols | inteiro | Gols marcados menos sofridos |
| pontos | inteiro | Pontuação final |
| aproveitamento_pct | decimal | Porcentagem dos pontos possíveis |
| gols_por_jogo | decimal | Média de gols marcados por jogo |


In [ ]:
output_file = Path("brasileirao_2024_2025.csv")
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print("Total de registros:", len(df))
print("Total de colunas:", len(df.columns))
print("Período: temporadas de 2024 e 2025")
print(f"Tamanho do CSV: {output_file.stat().st_size / 1024:.2f} KB")

## 5. ANÁLISE EXPLORATÓRIA
### 5.1 Estatísticas descritivas

In [ ]:
df.groupby("temporada")[["pontos", "gols_pro", "gols_contra", "aproveitamento_pct"]].agg(["mean", "median", "min", "max"]).round(2).rename(columns={"mean": "media", "median": "mediana", "min": "minimo", "max": "maximo"}, level=1)

### 5.2 Visualizações

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
current = df[df["temporada"] == 2025].sort_values("posicao")
axes[0].barh(current["clube"], current["pontos"], color="steelblue")
axes[0].invert_yaxis()
axes[0].set_title("Pontuação final do Brasileirão 2025")
axes[0].set_xlabel("Pontos")
variation = comparison.sort_values("diferenca_pontos")
axes[1].barh(variation.index, variation["diferenca_pontos"],
             color=["seagreen" if x >= 0 else "indianred" for x in variation["diferenca_pontos"]])
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Variação entre os clubes nas duas temporadas")
axes[1].set_xlabel("Pontos em 2025 menos pontos em 2024")
plt.tight_layout()
plt.show()

## 6. INSIGHTS
1. O Flamengo terminou 2025 em primeiro lugar, com 79 pontos e aproveitamento de 69.30%.
2. Entre os 16 clubes presentes nas duas temporadas, o Fluminense teve a maior melhora, com 18 pontos a mais. O Fortaleza teve a maior queda, com 25 pontos a menos.
3. O Flamengo marcou mais gols em 2025. Foram 78, uma média de 2.05 por partida.

## 7. LIMITAÇÕES
- A análise considera apenas duas temporadas e os totais finais dos clubes.
- Clubes promovidos ou rebaixados entre as temporadas não entram na comparação de evolução.
- A variação de pontos não explica, sozinha, as causas da melhora ou piora de um time.
- Mudanças no HTML da ESPN podem exigir ajustes no código de extração.

## 8. PRÓXIMOS PASSOS
- Incluir outras temporadas para verificar se as diferenças se repetem.
- Coletar resultados por partida para comparar jogos em casa e fora.

## 9. CÓDIGO
https://github.com/rafael-ifsp2/introducao-ciencia-dados-aula4
## 10. REFERÊNCIAS
- [ESPN — classificação do Brasileirão 2024](https://www.espn.com.br/futebol/classificacao/_/liga/bra.1/temporada/2024)
- [ESPN — classificação do Brasileirão 2025](https://www.espn.com.br/futebol/classificacao/_/liga/bra.1/temporada/2025)
